In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/18 16:41:36 WARN Utils: Your hostname, rog, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/11/18 16:41:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/18 16:41:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/18 16:41:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
from pyspark.sql import DataFrameReader
from pyspark.sql import functions as F, Window

# reading data from a CSV
data_reader = DataFrameReader(spark)
df = data_reader.csv("Pokemon.csv", header=True, )
df.show(3)

+---+---------+------+------+-----+---+------+-------+-------+-------+-----+----------+---------+
|  #|     Name|Type 1|Type 2|Total| HP|Attack|Defense|Sp. Atk|Sp. Def|Speed|Generation|Legendary|
+---+---------+------+------+-----+---+------+-------+-------+-------+-----+----------+---------+
|  1|Bulbasaur| Grass|Poison|  318| 45|    49|     49|     65|     65|   45|         1|    False|
|  2|  Ivysaur| Grass|Poison|  405| 60|    62|     63|     80|     80|   60|         1|    False|
|  3| Venusaur| Grass|Poison|  525| 80|    82|     83|    100|    100|   80|         1|    False|
+---+---------+------+------+-----+---+------+-------+-------+-------+-----+----------+---------+
only showing top 3 rows


In [3]:
# total pokemon in each type
(
df.groupBy(
    F.col("Type 1").alias("Primary Type"))
    .count()
    .show()
)

+------------+-----+
|Primary Type|count|
+------------+-----+
|       Water|  112|
|      Poison|   28|
|       Steel|   27|
|        Rock|   44|
|         Ice|   24|
|       Ghost|   32|
|       Fairy|   17|
|     Psychic|   57|
|      Dragon|   32|
|      Flying|    4|
|         Bug|   69|
|    Electric|   44|
|        Fire|   52|
|      Ground|   32|
|        Dark|   31|
|    Fighting|   27|
|       Grass|   70|
|      Normal|   98|
+------------+-----+



In [5]:
# total pokemon in each type
(
df.groupBy(
    F.col("Type 1").alias("Primary Type"))
    .count()
    .show()
)

+------------+-----+
|Primary Type|count|
+------------+-----+
|       Water|  112|
|      Poison|   28|
|       Steel|   27|
|        Rock|   44|
|         Ice|   24|
|       Ghost|   32|
|       Fairy|   17|
|     Psychic|   57|
|      Dragon|   32|
|      Flying|    4|
|         Bug|   69|
|    Electric|   44|
|        Fire|   52|
|      Ground|   32|
|        Dark|   31|
|    Fighting|   27|
|       Grass|   70|
|      Normal|   98|
+------------+-----+



In [ ]:
# total legendary pokemon in each type
(
df.filter(F.col("Legendary") == "True")
    .groupBy(
        F.col("Type 1").alias("Primary Type")
    )
    .count()
    .show()
)

+------------+-----+
|Primary Type|count|
+------------+-----+
|       Water|    4|
|       Steel|    4|
|        Rock|    4|
|         Ice|    2|
|       Ghost|    2|
|       Fairy|    1|
|     Psychic|   14|
|      Dragon|   12|
|      Flying|    2|
|    Electric|    4|
|        Fire|    5|
|      Ground|    4|
|        Dark|    2|
|       Grass|    3|
|      Normal|    2|
+------------+-----+



In [ ]:
# fasted pokemon per type
(
df.groupBy(
    F.col("Type 1"))
    .agg(
        F.max_by(F.col("Name"), F.col("Speed"))
    )
    .show(1000)
)

+--------+--------------------+
|  Type 1| max_by(Name, Speed)|
+--------+--------------------+
|     Bug|            Genesect|
|    Dark|             Yveltal|
|  Dragon|             Haxorus|
|Electric|          Electivire|
|   Fairy|             Xerneas|
|Fighting|            Primeape|
|    Fire|DarmanitanStandar...|
|  Flying|              Noibat|
|   Ghost| GourgeistSmall Size|
|   Grass|             Leafeon|
|  Ground|             Gliscor|
|     Ice|                Jynx|
|  Normal|            Raticate|
|  Poison|             Drapion|
| Psychic|            Sigilyph|
|    Rock|            Kabutops|
|   Steel|           Klinklang|
|   Water|              Swanna|
+--------+--------------------+



In [ ]:
(
df.groupBy(
    F.col("Type 1"))
    .agg(
        F.max_by(F.col("Name"), F.col("Speed")),
        F.max(F.col("Speed")) # trick to also show the max value
    )
    .show(1000)
)

+--------+--------------------+----------+
|  Type 1| max_by(Name, Speed)|max(Speed)|
+--------+--------------------+----------+
|     Bug|            Genesect|        99|
|    Dark|             Yveltal|        99|
|  Dragon|             Haxorus|        97|
|Electric|          Electivire|        95|
|   Fairy|             Xerneas|        99|
|Fighting|            Primeape|        95|
|    Fire|DarmanitanStandar...|        95|
|  Flying|              Noibat|        55|
|   Ghost| GourgeistSmall Size|        99|
|   Grass|             Leafeon|        95|
|  Ground|             Gliscor|        95|
|     Ice|                Jynx|        95|
|  Normal|            Raticate|        97|
|  Poison|             Drapion|        95|
| Psychic|            Sigilyph|        97|
|    Rock|            Kabutops|        80|
|   Steel|           Klinklang|        90|
|   Water|              Swanna|        98|
+--------+--------------------+----------+



In [ ]:
# fasted pokemon considering dual types
(
df.groupBy(
        F.col("Type 1"),
        F.col("Type 2")
    )
    .agg(
        F.max_by(F.col("Name"), F.col("Speed")),
        F.max(F.col("Speed")) # trick to also show the max value
    )
    .show(1000)
)

+--------+--------+--------------------+----------+
|  Type 1|  Type 2| max_by(Name, Speed)|max(Speed)|
+--------+--------+--------------------+----------+
|     Bug|    NULL|            Illumise|        85|
|     Bug|Electric|              Joltik|        65|
|     Bug|Fighting|           Heracross|        85|
|     Bug|    Fire|            Larvesta|        60|
|     Bug|  Flying|             Yanmega|        95|
|     Bug|   Ghost|            Shedinja|        40|
|     Bug|   Grass|            Leavanny|        92|
|     Bug|  Ground|             Nincada|        40|
|     Bug|  Poison|            Venomoth|        90|
|     Bug|    Rock|             Dwebble|        55|
|     Bug|   Steel|            Genesect|        99|
|     Bug|   Water|             Surskit|        65|
|    Dark|    NULL|               Absol|        75|
|    Dark|  Dragon|           Hydreigon|        98|
|    Dark|Fighting|             Scrafty|        58|
|    Dark|    Fire|            Houndoom|        95|
|    Dark|  